In [41]:
import pandas as pd
import sklearn
import numpy as np

In [42]:
import pandas as pd

df = pd.read_csv("./data/fra_with_season.csv", encoding='cp1252')
print(df.columns.tolist())
df = df[df["top_season"].notna() & (df["top_season"] != "none")].reset_index(drop=True)

base_dummies = df["Base"].fillna("").str.lower().str.get_dummies(sep=", ")
base_dummies.columns = ["base_" + col.strip().replace(" ", "_").replace("-", "_") for col in base_dummies.columns]
season_dummies = pd.get_dummies(df["top_season"], prefix="season", dtype=int)

# we are only using the base notes and then the seasonality data (what we are predicting)
data_encoded = pd.concat([base_dummies, season_dummies], axis=1)

print("Shape: ", data_encoded.shape)
data_encoded.head()
# we only have a 100ish columns for now because our data is currently getting web-scraped, we will re-run when we have all of them

['url', 'Perfume', 'Brand', 'Country', 'Gender', 'Rating Value', 'Rating Count', 'Year', 'Top', 'Middle', 'Base', 'Perfumer1', 'Perfumer2', 'mainaccord1', 'mainaccord2', 'mainaccord3', 'mainaccord4', 'mainaccord5', 'top_season']
Shape:  (197, 131)


,base_agarwood,base_agarwood_(oud),base_almond,base_amaretto,base_amber,base_ambergris,base_amberwood,base_ambrette,base_ambrette_(musk_mallow),base_ambroxan,...,base_white_wood,base_white_woods,base_woodsy_notes,base_woody_notes,base_ylang_ylang,base_ylang_ylang,season_fall,season_spring,season_summer,season_winter
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
2,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0


In [48]:
data_np_one_hot = data_encoded.to_numpy()
from sklearn.model_selection import train_test_split

# seperating data into x features and y (rear/front)
x = data_np_one_hot[:, :-1]
x_bias = np.hstack([x, np.ones((x.shape[0], 1))]) # add bias column to x
y = data_np_one_hot[:, -1]

[x_train, x_test, y_train, y_test] = train_test_split(x_bias, y, test_size = .3)

In [50]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    max_iter=200,
    random_state=42
)

mlp.fit(x_train, y_train)

y_prediction = mlp.predict(x_test)

correct = (y_prediction == y_test)
accuracy = np.mean(correct)
print(f"Accuracy: {accuracy*100:.2f}%")

Accuracy: 93.33%
